# **Install Required Packages**

bold textThis cell installs the latest versions of the `smolagents` and `litellm` Python packages. These libraries are used for building lightweight AI agents and interfacing with language models.

In [ ]:
!pip install smolagents -U  litellm

# **Initialize LiteLLM Model**

This cell imports the required classes from `smolagents` and initializes a `LiteLLMModel` using the Gemini 2.0 Flash model. The model is configured with a specific temperature and token limit for controlled generation.

 ⚠️ *Note: The API key should ideally be stored securely and not hardcoded.*


In [95]:
from smolagents import LiteLLMModel , CodeAgent

model = LiteLLMModel(
    "gemini/gemini-2.0-flash",
    temperature=0.2,
    api_key="",
    max_token = 8000
)


# **Exercise Search**

Searches for exercises targeting a specific body part using the ExerciseDB API from RapidAPI and returns the results as a JSON string containing exercise details.

In [82]:
from smolagents import Tool
import requests
import json

class ExerciseSearchTool(Tool):
    name = "exercise_search"
    description = (
        "Searches for exercises targeting a specific body part using the ExerciseDB API from RapidAPI "
        "and returns the results as a JSON string containing exercise details."
    )

    inputs = {
        "body_part": {
            "type": "string",
            "description": "The body part to search exercises for (e.g., 'back')."
        },
        "limit": {
            "type": "string",
            "description": "Maximum number of results to return (e.g., '10').",
            "default": "10",
            "nullable": True
        },
        "offset": {
            "type": "string",
            "description": "Starting point for results (e.g., '0').",
            "default": "0",
            "nullable": True
        }
    }

    output_type = "string"

    def __init__(self, api_key: str, **kwargs):
        super().__init__(**kwargs)
        self.api_key = api_key
        self.base_url = "https://exercisedb.p.rapidapi.com/exercises/bodyPart/{body_part}"
        self.headers = {
            "x-rapidapi-key": self.api_key,
            "x-rapidapi-host": "exercisedb.p.rapidapi.com"
        }

    def forward(self, body_part: str, limit: str = "10", offset: str = "0") -> str:
        """
        Search for exercises targeting the specified body part.

        Args:
            body_part (str): The body part to search for.
            limit (str): Maximum number of results (default: '10').
            offset (str): Starting point for results (default: '0').

        Returns:
            str: A JSON string of the exercise search results or an error message.
        """
        # Validate inputs
        if not body_part or not isinstance(body_part, str):
            return "Error: Body part must be a non-empty string."
        try:
            int(limit)
            int(offset)
        except ValueError:
            return "Error: Limit and offset must be valid integers."
        if int(limit) < 1:
            return "Error: Limit must be at least 1."
        if int(offset) < 0:
            return "Error: Offset must be non-negative."

        # Construct the URL with the body part
        url = self.base_url.format(body_part=body_part.lower().replace(" ", "%20"))

        # Construct the query parameters
        querystring = {
            "limit": limit,
            "offset": offset
        }

        try:
            # Send the GET request to the ExerciseDB API
            response = requests.get(url, headers=self.headers, params=querystring)
            response.raise_for_status()  # Raise an exception for HTTP errors
            return json.dumps(response.json(), indent=2)  # Return JSON as a formatted string
        except requests.RequestException as e:
            return f"Error performing exercise search: {str(e)}"
        except json.JSONDecodeError:
            return "Error: Invalid JSON response from the API."
        except Exception as e:
            return f"Error processing request: {str(e)}"

# **Exercise Body Part List**

Retrieves a list of available body parts for exercise searches using the ExerciseDB API from RapidAPI and returns the results as a JSON string.

In [83]:
from smolagents import Tool
import requests
import json

class ExerciseBodyPartListTool(Tool):
    name = "exercise_body_part_list"
    description = (
        "Retrieves a list of available body parts for exercise searches using the ExerciseDB API "
        "from RapidAPI and returns the results as a JSON string."
    )

    inputs = {}

    output_type = "string"

    def __init__(self, api_key: str, **kwargs):
        super().__init__(**kwargs)
        self.api_key = api_key
        self.url = "https://exercisedb.p.rapidapi.com/exercises/bodyPartList"
        self.headers = {
            "x-rapidapi-key": self.api_key,
            "x-rapidapi-host": "exercisedb.p.rapidapi.com"
        }

    def forward(self) -> str:
        """
        Retrieve the list of available body parts for exercise searches.

        Returns:
            str: A JSON string of the body part list or an error message.
        """
        try:
            # Send the GET request to the ExerciseDB API
            response = requests.get(self.url, headers=self.headers)
            response.raise_for_status()  # Raise an exception for HTTP errors
            return json.dumps(response.json(), indent=2)  # Return JSON as a formatted string
        except requests.RequestException as e:
            return f"Error retrieving body part list: {str(e)}"
        except json.JSONDecodeError:
            return "Error: Invalid JSON response from the API."
        except Exception as e:
            return f"Error processing request: {str(e)}"

# **Exercise Equipment List**

Retrieves a list of available equipment types for exercise searches using the ExerciseDB API from RapidAPI and returns the results as a JSON string.

In [84]:
from smolagents import Tool
import requests
import json

class ExerciseEquipmentListTool(Tool):
    name = "exercise_equipment_list"
    description = (
        "Retrieves a list of available equipment types for exercise searches using the ExerciseDB API "
        "from RapidAPI and returns the results as a JSON string."
    )

    inputs = {}

    output_type = "string"

    def __init__(self, api_key: str, **kwargs):
        super().__init__(**kwargs)
        self.api_key = api_key
        self.url = "https://exercisedb.p.rapidapi.com/exercises/equipmentList"
        self.headers = {
            "x-rapidapi-key": self.api_key,
            "x-rapidapi-host": "exercisedb.p.rapidapi.com"
        }

    def forward(self) -> str:
        """
        Retrieve the list of available equipment types for exercise searches.

        Returns:
            str: A JSON string of the equipment list or an error message.
        """
        try:
            # Send the GET request to the ExerciseDB API
            response = requests.get(self.url, headers=self.headers)
            response.raise_for_status()  # Raise an exception for HTTP errors
            return json.dumps(response.json(), indent=2)  # Return JSON as a formatted string
        except requests.RequestException as e:
            return f"Error retrieving equipment list: {str(e)}"
        except json.JSONDecodeError:
            return "Error: Invalid JSON response from the API."
        except Exception as e:
            return f"Error processing request: {str(e)}"

In [94]:
api_key = ""
excecise_search = ExerciseSearchTool(api_key)
available_ex = ExerciseBodyPartListTool(api_key)
eqipment_av = ExerciseEquipmentListTool(api_key)
research_agent = CodeAgent(
    model=model,
    tools=[
        excecise_search,
        available_ex,
        eqipment_av
    ],
    name="research_agent",    verbosity_level=2,
    max_steps=10,
    additional_authorized_imports=['math', 'statistics', 'datetime', 'collections', 'queue', 'random', 're',
'unicodedata', 'itertools', 'time', 'stat' , 'json']
)

#  **Define Fitness Consultant Query**

This cell sets a user-defined product query string representing the customer's request. This text will be used to guide the AI agent’s  fitness consultant logic.  

In [86]:
query = "a high intensite back muscel"

# **Craft Agent Prompt for Recommendation Task**

  This cell creates a detailed task prompt for the AI agent. It simulates the role of an expert fitness consultant agent and outlines step-by-step methodology evidence-based exercise program tailored to the user's specific goals, incorporating appropriate exercises, equipment recommendations, and structured progression .

In [90]:
task  = f"""Act as an expert fitness consultant analyzing '{query}' to create a personalized exercise program.

## PRIMARY OBJECTIVE:
Generate a comprehensive, evidence-based exercise program tailored to the user's specific goals, incorporating appropriate exercises, equipment recommendations, and structured progression.

## ANALYSIS METHODOLOGY:

1. GOAL INTERPRETATION:
   - Analyze the user's query to identify specific fitness objectives:
     * Body composition goals (muscle building, fat loss, toning)
     * Performance goals (strength, endurance, flexibility, power)
     * Aesthetic targets (specific body parts/muscle groups)
     * Health objectives (improved posture, injury prevention, rehabilitation)
   - Classify primary and secondary goals to prioritize exercise selection

2. BODY PART MAPPING:
   - Identify target muscle groups based on user's goals
   - Create comprehensive mapping of all relevant body parts:
     * UPPER BODY: Chest, Back, Shoulders, Biceps, Triceps, Forearms
     * CORE: Abdominals, Obliques, Lower Back
     * LOWER BODY: Quadriceps, Hamstrings, Glutes, Calves, Adductors, Abductors
     * FUNCTIONAL: Rotator Cuff, Hip Flexors, Spinal Erectors
   - Determine optimal training frequency for each muscle group

3. EQUIPMENT ASSESSMENT:
   - Compile appropriate equipment recommendations from all available options:
     * FREE WEIGHTS: Dumbbells, Barbells, Kettlebells, Weight Plates
     * MACHINES: Cable Machines, Smith Machine, Leg Press, Chest Press
     * BODYWEIGHT: Pull-up Bar, Dip Station, Suspension Trainers
     * CARDIO: Treadmill, Elliptical, Stationary Bike, Rowing Machine
     * ACCESSORIES: Resistance Bands, Medicine Balls, Foam Rollers, Stability Balls
   - Prioritize versatile equipment that maximizes exercise variety

4. EXERCISE SELECTION PROCESS:
   - For each target muscle group/body part:
     * Identify 3-5 optimal exercises based on scientific efficacy
     * Include compound and isolation movements in appropriate ratios
     * Specify correct form cues and technique guidelines
     * Note common errors to avoid
     * Provide appropriate progression and regression options

5. PROGRAM STRUCTURE DEVELOPMENT:
   - Design optimal training split based on time availability and recovery needs:
     * Full Body, Upper/Lower, Push/Pull/Legs, or Body Part Split
   - Establish appropriate volume (sets/reps) and intensity guidelines
   - Create periodization model for progressive overload
   - Include detailed warm-up and cooldown protocols
   - Specify rest intervals between sets and exercises

6. MULTIMEDIA RESOURCE CURATION:
   - For each recommended exercise:
     * Locate high-quality demonstration videos
     * Find visual form guides and technical breakdowns
     * Link to scientific studies supporting efficacy where available

## OUTPUT FORMAT:

1. EXECUTIVE SUMMARY:
   - Brief overview of recommended program approach
   - Key principles guiding the recommendations
   - Expected timeline for results

2. GOAL ANALYSIS:
   - Interpretation of user's specific goals
   - Explanation of physiological requirements to achieve these goals
   - Realistic expectations and benchmarks

3. PROGRAM OVERVIEW:
   - Recommended training split with weekly schedule
   - Progression model over 4-12 weeks
   - Volume and intensity guidelines

4. DETAILED WORKOUT PLANS:
   - Day-by-day exercise recommendations
   - Specific sets, reps, and rest periods
   - Proper warm-up sequences

5. EXERCISE LIBRARY (organized by body part):
   - For each exercise:
     * Clear exercise name and classification
     * Primary and secondary muscles worked
     * Required equipment
     * Detailed form instructions
     * Video demonstration link
     * Common mistakes and corrections
     * Progression/regression options

6. EQUIPMENT RECOMMENDATIONS:
   - Essential equipment list with alternatives
   - Budget-friendly options
   - Space-efficient solutions

7. TRACKING & PROGRESSION:
   - Specific metrics to track
   - When and how to increase difficulty
   - Deload strategies
8. All available gifurl
## QUALITY STANDARDS:
- Base all recommendations on peer-reviewed exercise science
- Prioritize evidence-based approaches over fitness trends
- Consider individual factors (implied experience level, potential limitations)
- Balance optimal approaches with practical implementation
- Provide specific, actionable guidance rather than generic advice
- Include safety considerations and injury prevention strategies
"""

In [ ]:
result = research_agent.run(task)

In [93]:
from IPython.display import Markdown, display

display(Markdown(result))



**Executive Summary:**

This program is designed to build back muscle using high-intensity compound exercises. It focuses on stimulating muscle growth through a combination of barbell and cable exercises.

**Goal Analysis:**

The user's goal is to build back muscle. This requires stimulating muscle protein synthesis through resistance training and adequate nutrition. High-intensity training with compound exercises is effective for achieving this goal.

**Program Overview:**

*   **Training Split:** Full Body (can be incorporated into a full-body routine or performed as a dedicated back workout)
*   **Progression Model:** Linear Progression (increase weight when able to perform 12 reps with good form)
*   **Volume and Intensity:** 3 sets of 8-12 reps, RPE 7-8 (challenging but maintainable)

**Detailed Workout Plan:**

*   **Warm-up:** 5 minutes of light cardio (e.g., jumping jacks, arm circles) followed by dynamic stretching (e.g., torso twists, arm swings).
*   **Workout:**
    *   Barbell Bent Over Row: 3 sets of 8-12 reps, 60-90 seconds rest
    *   Alternate Lateral Pulldown: 3 sets of 8-12 reps per side, 60-90 seconds rest
    *   Barbell Pullover to Press: 3 sets of 8-12 reps, 60-90 seconds rest
*   **Cool-down:** 5 minutes of static stretching (e.g., lat stretch, hamstring stretch).

**Exercise Library:**

*   **Barbell Bent Over Row**
    *   **Classification:** Compound
    *   **Muscles Worked:** Upper Back (primary), Biceps, Forearms (secondary)
    *   **Equipment:** Barbell
    *   **Form Instructions:** Stand with feet shoulder-width apart, bend at the hips keeping back straight. Pull the barbell to your lower chest.
    *   **Video Demonstration:** [https://v2.exercisedb.io/image/NBNo-FtIYrxir4](https://v2.exercisedb.io/image/NBNo-FtIYrxir4)
    *   **Common Mistakes:** Rounding the back, using momentum.
    *   **Progression/Regression:** Increase weight, use lighter weight or resistance band.
*   **Alternate Lateral Pulldown**
    *   **Classification:** Compound
    *   **Muscles Worked:** Lats (primary), Biceps, Rhomboids (secondary)
    *   **Equipment:** Cable Machine
    *   **Form Instructions:** Sit at the cable machine, pull the handle to your chest, alternating sides.
    *   **Video Demonstration:** [https://v2.exercisedb.io/image/u0pgfxbbnrNcEa](https://v2.exercisedb.io/image/u0pgfxbbnrNcEa)
    *   **Common Mistakes:** Leaning back too far, using momentum.
    *   **Progression/Regression:** Increase weight, use lighter weight or resistance band.
*   **Barbell Pullover to Press**
    *   **Classification:** Compound
    *   **Muscles Worked:** Lats (primary), Triceps, Chest, Shoulders (secondary)
    *   **Equipment:** Barbell
    *   **Form Instructions:** Lie on a bench, lower the barbell behind your head, then press back up.
    *   **Video Demonstration:** [https://v2.exercisedb.io/image/EbWMiOyCKTdpBB](https://v2.exercisedb.io/image/EbWMiOyCKTdpBB)
    *   **Common Mistakes:** Bending the arms too much, using too much weight.
    *   **Progression/Regression:** Increase weight, use lighter weight or dumbbell.

**Equipment Recommendations:**

*   Barbell
*   Weight Plates
*   Cable Machine

**Tracking & Progression:**

*   Track the weight used for each exercise and the number of reps completed.
*   Increase the weight when you can perform 12 reps with good form.
*   Deload every 4-6 weeks by reducing the weight by 20-30%.
